# VisioLex — Training on Google Colab

**Runtime → Change runtime type → T4 GPU** before running.

This notebook:
1. Clones the VisioLex repo and installs dependencies
2. Mounts Google Drive for checkpoint persistence
3. Downloads GRID Corpus (single speaker for quick test, all 34 for full run)
4. Pre-processes mouth crops with MediaPipe
5. Trains VisioLex and saves checkpoints to Drive


In [ ]:
# ── 1. Check GPU ──────────────────────────────────────────────────────────
!nvidia-smi

In [ ]:
# ── 2. Mount Google Drive ─────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/visiolex'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive directory: {DRIVE_DIR}')

In [ ]:
# ── 3. Clone repo & install dependencies ─────────────────────────────────
# Replace with your actual repo URL after pushing to GitHub
REPO_URL = 'https://github.com/YOUR_USERNAME/visiolex.git'

%cd /content
!git clone {REPO_URL} visiolex 2>/dev/null || (cd visiolex && git pull)
%cd /content/visiolex

!pip install -q -r requirements.txt
print('Dependencies installed.')

In [ ]:
# ── 4. Download GRID Corpus ───────────────────────────────────────────────
# GRID requires registration at https://spandh.dcs.shef.ac.uk/gridcorpus/
# After registration you get direct download links for each speaker.
#
# For a quick smoke-test we show speaker s1 only.
# Replace S1_VIDEO_URL and S1_ALIGN_URL with your download links.

import os

GRID_ROOT = '/content/visiolex/data/grid'
os.makedirs(f'{GRID_ROOT}/s1/video/mpg_6000', exist_ok=True)
os.makedirs(f'{GRID_ROOT}/s1/align', exist_ok=True)

# ---- Replace these URLs with actual GRID download links ----
# S1_VIDEO_URL = 'https://...'
# S1_ALIGN_URL  = 'https://...'
# !wget -q -O /tmp/s1_video.zip {S1_VIDEO_URL}
# !wget -q -O /tmp/s1_align.zip  {S1_ALIGN_URL}
# !unzip -q /tmp/s1_video.zip -d {GRID_ROOT}/s1/video/
# !unzip -q /tmp/s1_align.zip  -d {GRID_ROOT}/s1/
# ------------------------------------------------------------

print('GRID data ready (update URLs above for your registration).')

In [ ]:
# ── 5. Pre-process mouth crops ────────────────────────────────────────────
!python scripts/preprocess_grid.py \
    --grid_root data/grid \
    --processed_dir data/processed \
    --speakers 1

In [ ]:
# ── 6. Train VisioLex ─────────────────────────────────────────────────────
# Single speaker, 20 epochs — just to validate the pipeline.
# For the full model: remove --speakers, set --epochs 50.

!python scripts/train.py \
    --config configs/train.yaml \
    --grid_root data/grid \
    --processed_dir data/processed \
    --speakers 1 \
    --epochs 20 \
    --batch_size 16 \
    --no_wandb

In [ ]:
# ── 7. Copy best checkpoint to Drive ─────────────────────────────────────
import shutil, os

src = '/content/visiolex/checkpoints/best.pt'
dst = f'{DRIVE_DIR}/best.pt'
if os.path.exists(src):
    shutil.copy2(src, dst)
    print(f'Checkpoint saved to {dst}')
else:
    print('No checkpoint found — did training complete?')

In [ ]:
# ── 8. Quick inference test ───────────────────────────────────────────────
import sys
sys.path.insert(0, '/content/visiolex')

import torch
from src.models import VisioLexModel
from src.decoding import GreedyCTCDecoder
from src.utils.text import VOCAB_SIZE, BLANK_IDX

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = VisioLexModel(vocab_size=VOCAB_SIZE).to(device)
ckpt = torch.load('/content/visiolex/checkpoints/best.pt', map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

decoder = GreedyCTCDecoder(blank_idx=BLANK_IDX)

# Dummy input — replace with a real pre-processed clip
dummy = torch.randn(1, 1, 75, 64, 64, device=device)
with torch.no_grad():
    log_probs = model(dummy)

print('Prediction (random weights):', decoder.decode(log_probs[:, 0, :]))
print('Val WER from checkpoint:', ckpt.get('val_wer', 'N/A'))